<a href="https://colab.research.google.com/github/contemplator/nchu-research-method-traffic-violation-dataset/blob/feat/152-images-leo/notebooks/NCHU_Research_YOLOv7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **國立中興大學 研究方法 (Research Methodology) - 期末專案**
## **Part 2: YOLOv7 台灣市區交通設施與違規標誌即時偵測與消融研究**

---
### 📌 **專案基本資訊**
* **專題題目**：基於 YOLOv7 深度學習之台灣多類別交通設施與違規標誌即時偵測與動態影片分析
* **繳交期限 (Deadline)**：**2026/12/17**
* **成果繳交格式**：`{StudentID1}_{StudentID2}_{StudentID3}.zip`（含報告 Slides / 文件）
* **6 大標註類別 (Target Classes)**：
  0. `traffic_light` (交通號誌/紅綠燈)
  1. `crosswalk` (行人穿越道/斑馬線)
  2. `two_stage_left_turn_box` (機車兩段式左轉待轉區)
  3. `Sign_follow` (遵行標誌)
  4. `Sign_prohibit` (禁制標誌)
  5. `Sign_limit` (限制標誌)

* **模型倉庫**：`contemplator/nchu-research-method-traffic-violation-model`
* **資料/實驗倉庫**：`contemplator/nchu-research-method-traffic-violation-dataset`


# **Step 1: 環境檢查與 GPU 配置 (Environment Checking)**

In [1]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current Device: {torch.cuda.get_device_name(0)}")

Thu Oct 15 05:32:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **Step 2: 掛載 Google 雲端硬碟 (Mount Google Drive)**

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# **Step 3: 下載專案倉庫與安裝 YOLOv7 依賴 (Setup YOLOv7)**
> 若為 Private 倉庫，可填寫您的 GitHub Personal Access Token (PAT) 以便免密碼 Clone。

In [5]:
%cd /content
!mkdir -p workspace
%cd /content/workspace

# Clone 倉庫 (若已存在加上判斷避免報錯)
![ ! -d "yolov7" ] && git clone https://github.com/contemplator/nchu-research-method-traffic-violation-model.git yolov7
![ ! -d "dataset_repo" ] && git clone -b feat/152-images-leo https://github.com/contemplator/nchu-research-method-traffic-violation-dataset.git dataset_repo
!ln -sfn /content/workspace/dataset_repo /content/workspace/nchu-research-method-traffic-violation-dataset

# 進入 yolov7 並只安裝 Colab 缺少的套件 (避免降級 numpy 造成編譯失敗)
%cd /content/workspace/yolov7
!pip install thop albumentations

import os

# 1. 自動修復 NumPy 2.x 相容性問題 (NumPy 2.0+ 移除 np.trapz 改為 np.trapezoid)
metrics_file = "/content/workspace/yolov7/utils/metrics.py"
if os.path.exists(metrics_file):
    with open(metrics_file, "r", encoding="utf-8") as f:
        content = f.read()
    if "np.trapz" in content:
        content = content.replace("np.trapz", "getattr(np, \"trapezoid\", getattr(np, \"trapz\", None))")
        with open(metrics_file, "w", encoding="utf-8") as f:
            f.write(content)
        print("✅ 已成功修復 yolov7/utils/metrics.py 的 NumPy 2.x 相容性！")

# 2. 自動修復 google_utils.py 在 GitHub API 限流或找不到本地路徑時引發 KeyError: "assets" 的問題
google_utils_file = "/content/workspace/yolov7/utils/google_utils.py"
if os.path.exists(google_utils_file):
    with open(google_utils_file, "r", encoding="utf-8") as f:
        g_content = f.read()
    if "response['assets']" in g_content:
        g_content = g_content.replace("response['assets']", "response.get('assets', [])")
        with open(google_utils_file, "w", encoding="utf-8") as f:
            f.write(g_content)
        print("✅ 已成功修復 yolov7/utils/google_utils.py 的 GitHub API 防呆機制！")


/content
/content/workspace
/content/workspace/yolov7


## **下載官方預訓練權重 (Pretrained Weights)**

In [6]:
%cd /content/workspace/yolov7
!wget -nc -P /content/workspace/yolov7 https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt
!wget -nc -P /content/workspace/yolov7 https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt

/content/workspace/yolov7
--2026-10-15 06:18:00--  https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/511187726/ba7d01ee-125a-4134-8864-fa1abcbf94d5?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-15T07%3A02%3A31Z&rscd=attachment%3B+filename%3Dyolov7-tiny.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-10-15T06%3A01%3A44Z&ske=2026-10-15T07%3A02%3A31Z&sks=b&skv=2018-11-09&sig=631ZykpOIAdHJxjFgbSaxrqxnIWg6uoaySzAdR%2Fdjzk%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5MjA0Njg4MSwibmJmIjoxNzkyMDQ1MDgxLCJwYXRoIjoicmVsZWFzZWFz

# **Step 4: 準備資料集 (Dataset Preparation)**
將本地打包切分好的 `dataset.zip` 自 Google Drive 解壓縮至工作區：

In [7]:
import os
# 請確認 Drive 上的 zip 路徑是否正確 (例如 Drive 中的 NCHU_YOLO_Project_Traffic_Facility_6_classes/dataset.zip)
DRIVE_DATASET_ZIP = "/content/drive/MyDrive/NCHU_YOLO_Project_Traffic_Facility_6_classes/dataset.zip"
TARGET_DATA_DIR = "/content/workspace/dataset_repo/data"

if os.path.exists(DRIVE_DATASET_ZIP):
    !unzip -o -q {DRIVE_DATASET_ZIP} -d {TARGET_DATA_DIR}
    print("✅ 資料集自 Google Drive 解壓縮完成！")
    !ls -la /content/workspace/dataset_repo/data/dataset
else:
    print(f"⚠️ 尚未在 {DRIVE_DATASET_ZIP} 找到壓縮檔，若資料已存在於 dataset_repo 中則可直接使用。")

# 檢查 data.yaml
!cat /content/workspace/dataset_repo/configs/traffic_facility.yaml

✅ 資料集自 Google Drive 解壓縮完成！
total 20
drwxr-xr-x 5 root root 4096 Oct 15 03:54 .
drwxr-xr-x 4 root root 4096 Oct 15 06:18 ..
-rw-r--r-- 1 root root    0 Oct  6 07:25 .gitkeep
drwxr-xr-x 4 root root 4096 Oct 15 03:54 test
drwxr-xr-x 4 root root 4096 Oct 15 03:54 train
drwxr-xr-x 4 root root 4096 Oct 15 03:54 val
# NCHU Research Method - Taiwan Traffic Violation Dataset Config
# 4 Classes: red_line_parking, double_parking, yellow_box_parking, sidewalk_parking

# Dataset paths: Relative to execution directory or absolute paths
# When training in yolov7 directory, paths point to dataset repo:
train: ../nchu-research-method-traffic-violation-dataset/data/dataset/train/images
val: ../nchu-research-method-traffic-violation-dataset/data/dataset/val/images
test: ../nchu-research-method-traffic-violation-dataset/data/dataset/test/images

# Number of classes
nc: 4

# Class names
names: ['red_line_parking', 'double_parking', 'yellow_box_parking', 'sidewalk_parking']


# **Step 5: 4 組消融實驗訓練 (Ablation Experiments)**

| 組別 | 模型架構 | Epochs | Batch | 目的與探討目標 |
| :---: | :--- | :---: | :---: | :--- |
| **Exp 1** | YOLOv7-tiny | 50 | 16 | 輕量輕量模型基準線 (Baseline) |
| **Exp 2** | YOLOv7 Base | 50 | 8 | 骨幹模型尺度對複雜遮擋與小目標效益 |
| **Exp 3** | YOLOv7-tiny | 100 | 16 | 延長訓練週期之充分收斂性與過擬合檢視 |
| **Exp 4** | YOLOv7-tiny (Tuned) | 100 | 16 | 超參數調校 (Mosaic 增強 + 學習率微調) |

### **🔥 Exp 1: YOLOv7-tiny 輕量基準線 (50 Epochs)**

> 加入區塊引述符號



In [12]:
import os

patch_code = """import torch
_orig_load = torch.load
torch.load = lambda *args, **kwargs: _orig_load(*args, **{**kwargs, 'weights_only': False})
"""

for fname in ['train.py', 'test.py', 'detect.py']:
    fpath = os.path.join('/content/workspace/yolov7', fname)
    if os.path.exists(fpath):
        with open(fpath, 'r', encoding='utf-8') as f:
            content = f.read()
        if 'weights_only' not in content:
            with open(fpath, 'w', encoding='utf-8') as f:
                f.write(patch_code + content)

print("✅ PyTorch 2.6 相容性修補完成！")


✅ PyTorch 2.6 相容性修補完成！


In [14]:
# 停用 wandb 互動提示
%env WANDB_MODE=disabled

%cd /content/workspace/yolov7
!rm -rf runs/train/exp1_yolov7_tiny_baseline*
!python train.py \
  --workers 4 \
  --device 0 \
  --batch-size 16 \
  --epochs 50 \
  --data /content/workspace/dataset_repo/configs/traffic_facility.yaml \
  --img 640 640 \
  --cfg /content/workspace/dataset_repo/configs/yolov7-tiny-custom.yaml \
  --weights yolov7-tiny.pt \
  --name exp1_yolov7_tiny_baseline \
  --hyp data/hyp.scratch.tiny.yaml \
  --exist-ok

env: WANDB_MODE=disabled
/content/workspace/yolov7
I0000 00:00:1792045780.335022   15788 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792045782.544808   15788 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
YOLOR 🚀 a207844 torch 2.11.0+cu130 CUDA:0 (Tesla T4, 14912.6875MB)

Namespace(weights='yolov7-tiny.pt', cfg='/content/workspace/dataset_repo/configs/yolov7-tiny-custom.yaml', data='/content/workspace/dataset_repo/configs/traffic_facility.yaml', hyp='data/hyp.scratch.tiny.yaml', epochs=50, batch_size=16, img_size=[640, 640], rect=False, resume=False, nosave=False, notest=False, noautoanchor=False, evolve=False, bucket='', cache_images=False, image_weights=False, device='0', multi_scale=False, single_cls=False, adam=Fa

### **🔥 Exp 2: YOLOv7 Base 尺度對比 (50 Epochs)**

In [17]:
%cd /content/workspace/yolov7
!rm -rf runs/train/exp2_yolov7_base_scale*
!python train.py \
  --workers 4 \
  --device 0 \
  --batch-size 8 \
  --epochs 50 \
  --data /content/workspace/dataset_repo/configs/traffic_facility.yaml \
  --img 640 640 \
  --cfg cfg/training/yolov7.yaml \
  --weights yolov7.pt \
  --name exp2_yolov7_base_scale \
  --hyp data/hyp.scratch.p5.yaml \
  --exist-ok

env: WANDB_MODE=disabled
/content/workspace/yolov7
I0000 00:00:1792046048.128509   17028 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792046050.510522   17028 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
YOLOR 🚀 a207844 torch 2.11.0+cu130 CUDA:0 (Tesla T4, 14912.6875MB)

Namespace(weights='yolov7.pt', cfg='cfg/training/yolov7.yaml', data='/content/workspace/dataset_repo/configs/traffic_facility.yaml', hyp='data/hyp.scratch.p5.yaml', epochs=50, batch_size=8, img_size=[640, 640], rect=False, resume=False, nosave=False, notest=False, noautoanchor=False, evolve=False, bucket='', cache_images=False, image_weights=False, device='0', multi_scale=False, single_cls=False, adam=False, sync_bn=False, local_rank=-1, workers=4, p

### **🔥 Exp 3: 充分收斂週期對比 (100 Epochs)**

In [18]:
%cd /content/workspace/yolov7
!rm -rf runs/train/exp3_yolov7_tiny_100ep*
!python train.py \
  --workers 4 \
  --device 0 \
  --batch-size 16 \
  --epochs 100 \
  --data /content/workspace/dataset_repo/configs/traffic_facility.yaml \
  --img 640 640 \
  --cfg /content/workspace/dataset_repo/configs/yolov7-tiny-custom.yaml \
  --weights yolov7-tiny.pt \
  --name exp3_yolov7_tiny_100ep \
  --hyp data/hyp.scratch.tiny.yaml \
  --exist-ok

/content/workspace/yolov7
I0000 00:00:1792046384.329614   18558 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792046386.873174   18558 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
YOLOR 🚀 a207844 torch 2.11.0+cu130 CUDA:0 (Tesla T4, 14912.6875MB)

Namespace(weights='yolov7-tiny.pt', cfg='/content/workspace/dataset_repo/configs/yolov7-tiny-custom.yaml', data='/content/workspace/dataset_repo/configs/traffic_facility.yaml', hyp='data/hyp.scratch.tiny.yaml', epochs=100, batch_size=16, img_size=[640, 640], rect=False, resume=False, nosave=False, notest=False, noautoanchor=False, evolve=False, bucket='', cache_images=False, image_weights=False, device='0', multi_scale=False, single_cls=False, adam=False, sync_bn=False, loca

### **🔥 Exp 4: 超參數與資料增強微調 (Custom Hyp + 100 Epochs)**

In [19]:
%cd /content/workspace/yolov7
!rm -rf runs/train/exp4_yolov7_tiny_tuned*
!python train.py \
  --workers 4 \
  --device 0 \
  --batch-size 16 \
  --epochs 100 \
  --data /content/workspace/dataset_repo/configs/traffic_facility.yaml \
  --img 640 640 \
  --cfg /content/workspace/dataset_repo/configs/yolov7-tiny-custom.yaml \
  --weights yolov7-tiny.pt \
  --name exp4_yolov7_tiny_tuned \
  --hyp /content/workspace/dataset_repo/configs/hyp.scratch.custom.yaml \
  --exist-ok

/content/workspace/yolov7
I0000 00:00:1792046601.804932   19562 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792046604.090676   19562 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
YOLOR 🚀 a207844 torch 2.11.0+cu130 CUDA:0 (Tesla T4, 14912.6875MB)

Namespace(weights='yolov7-tiny.pt', cfg='/content/workspace/dataset_repo/configs/yolov7-tiny-custom.yaml', data='/content/workspace/dataset_repo/configs/traffic_facility.yaml', hyp='/content/workspace/dataset_repo/configs/hyp.scratch.custom.yaml', epochs=100, batch_size=16, img_size=[640, 640], rect=False, resume=False, nosave=False, notest=False, noautoanchor=False, evolve=False, bucket='', cache_images=False, image_weights=False, device='0', multi_scale=False, single_cls=F

# **Step 6: 測試集獨立指標評估 (Test Evaluation on `test` split)**
直接使用未曾參與訓練的獨立測試集 (`test/images` 與 `test/labels`) 評估模型泛化性能。
* 加上 `--task test`：強制以測試集資料進行評估（避免預設使用驗證集 val）。
* 加上 `--verbose`：輸出各類別詳細指標表格（Precision, Recall, mAP@0.5, mAP@0.5:0.95）。
* 加上 `--save-txt --save-conf`：匯出預測邊界框與信心度檔案至 `runs/test/`。


In [ ]:
import glob, os
%cd /content/workspace/yolov7

# 清空舊的測試評估輸出，避免 --exist-ok 造成結果殘留
!rm -rf runs/test/exp4_test_evaluation*

# 自動尋找最新的 exp4 訓練權重 (相容有無加上數字後綴的資料夾)
exp4_weights = sorted(glob.glob("runs/train/exp4_yolov7_tiny_tuned*/weights/best.pt"), key=os.path.getmtime)
BEST_WEIGHT = exp4_weights[-1] if exp4_weights else "runs/train/exp4_yolov7_tiny_tuned/weights/best.pt"
print(f"🎯 使用權重進行測試集評估: {BEST_WEIGHT}")

# 評估於測試集 (test split) 之獨立表現 (開啟 --task test, --verbose, --save-txt, --save-conf, --exist-ok)
!python test.py \
  --data /content/workspace/dataset_repo/configs/traffic_facility.yaml \
  --img 640 \
  --batch 16 \
  --conf 0.001 \
  --iou 0.65 \
  --device 0 \
  --weights {BEST_WEIGHT} \
  --name exp4_test_evaluation \
  --task test \
  --verbose \
  --save-txt \
  --save-conf \
  --exist-ok

# **Step 7: 測試集圖片推論與視覺化輸出 (Test Image Inference)**
* 將獨立測試集圖片 (`test/images/`) 畫上預測邊界框與類別信心度，輸出高解析度 `.jpg` 圖片至 `runs/detect/detect_test_images/`。
* 加上 `--save-txt --save-conf --exist-ok`：自動儲存每個檢出目標的正規化座標與信心值文字檔至 `runs/detect/detect_test_images/labels/`，供 Step 8 與標準答案逐一比對。


In [ ]:
import glob, os
%cd /content/workspace/yolov7

# 清空舊的推論輸出，避免 --exist-ok 造成 labels/*.txt 重複附加舊預測框
!rm -rf runs/detect/detect_test_images*

exp4_weights = sorted(glob.glob("runs/train/exp4_yolov7_tiny_tuned*/weights/best.pt"), key=os.path.getmtime)
BEST_WEIGHT = exp4_weights[-1] if exp4_weights else "runs/train/exp4_yolov7_tiny_tuned/weights/best.pt"
print(f"🎯 使用權重進行測試集圖片推論: {BEST_WEIGHT}")

# 測試集圖片推論 (加上 --save-txt --save-conf --exist-ok 自動匯出標籤座標與信心度文字檔)
!python detect.py \
  --weights {BEST_WEIGHT} \
  --conf 0.25 \
  --img-size 640 \
  --source /content/workspace/dataset_repo/data/dataset/test/images/ \
  --name detect_test_images \
  --save-txt \
  --save-conf \
  --exist-ok


# **Step 8: 成果圖表與可解釋性誤差診斷 (Interpretability & Error Analysis)**
系統性比對「真實標籤 (Ground Truth)」與「模型預測 (Predictions)」：
1. **混淆矩陣與 PR 曲線**：全覽分類混淆關係。
2. **抓到 / 漏抓 / 誤判逐張診斷**：深入剖析哪些目標成功辨識 (TP)、哪些目標漏抓 (FN) 或虛警誤判 (FP)。
3. **真值 vs 預測並列對比圖**：以可解釋視覺化佐證報告結論。


In [ ]:
import os
import glob
import matplotlib.pyplot as plt
import cv2

# 1. 顯示測試集混淆矩陣 (Confusion Matrix)
cm_candidates = sorted(glob.glob("/content/workspace/yolov7/runs/test/exp4_test_evaluation*/confusion_matrix.png"), key=os.path.getmtime)
if not cm_candidates:
    cm_candidates = sorted(glob.glob("/content/workspace/yolov7/runs/train/exp4_yolov7_tiny_tuned*/confusion_matrix.png"), key=os.path.getmtime)

if cm_candidates:
    test_cm = cm_candidates[-1]
    plt.figure(figsize=(10, 8), dpi=150)
    img = cv2.imread(test_cm)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Test Set Confusion Matrix ({os.path.basename(os.path.dirname(test_cm))})", fontsize=14)
    plt.axis("off")
    plt.show()

# 2. 顯示 Precision-Recall 曲線 (PR Curve)
pr_candidates = sorted(glob.glob("/content/workspace/yolov7/runs/test/exp4_test_evaluation*/PR_curve.png"), key=os.path.getmtime)
if pr_candidates:
    test_pr = pr_candidates[-1]
    plt.figure(figsize=(9, 7), dpi=150)
    img = cv2.imread(test_pr)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Precision-Recall Curve (各類別 PR 曲線)", fontsize=14)
    plt.axis("off")
    plt.show()

In [ ]:
# 3. 系統性誤差診斷：逐一比對 test/labels 與 detect 預測結果 (抓到 TP / 漏抓 FN / 誤判 FP)
import glob, os
%cd /content/workspace/dataset_repo

pred_dirs = sorted(glob.glob("/content/workspace/yolov7/runs/detect/detect_test_images*/labels"), key=os.path.getmtime)
PRED_LABEL_DIR = pred_dirs[-1] if pred_dirs else "/content/workspace/yolov7/runs/detect/detect_test_images/labels"
print(f"📂 比對預測標籤目錄: {PRED_LABEL_DIR}")

!python scripts/analyze_test_results.py \
  --gt data/dataset/test/labels \
  --pred {PRED_LABEL_DIR} \
  --images data/dataset/test/images \
  --classes data/dataset/classes.txt \
  --iou 0.5

In [ ]:
# 4. 顯示 test.py 產生的測試批次「真值 (Labels)」與「預測 (Pred)」並列對比圖
import glob, os
pred_batches = sorted(glob.glob("/content/workspace/yolov7/runs/test/exp4_test_evaluation*/test_batch0_pred.jpg"), key=os.path.getmtime)
label_batches = sorted(glob.glob("/content/workspace/yolov7/runs/test/exp4_test_evaluation*/test_batch0_labels.jpg"), key=os.path.getmtime)

if label_batches and pred_batches:
    fig, axes = plt.subplots(1, 2, figsize=(22, 11), dpi=150)
    img_lbl = cv2.imread(label_batches[-1])
    img_pred = cv2.imread(pred_batches[-1])
    
    axes[0].imshow(cv2.cvtColor(img_lbl, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Ground Truth (測試集標準答案)", fontsize=14)
    axes[0].axis("off")
    
    axes[1].imshow(cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Model Predictions (模型檢測預測結果)", fontsize=14)
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    # 若無批次圖則展示推論成果抽樣網格 (2x3)
    det_images = sorted(glob.glob("/content/workspace/yolov7/runs/detect/detect_test_images*/*.jpg"))[:6]
    if det_images:
        fig, axes = plt.subplots(2, 3, figsize=(16, 10), dpi=150)
        axes = axes.flatten()
        for i, img_path in enumerate(det_images):
            img = cv2.imread(img_path)
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            axes[i].set_title(os.path.basename(img_path), fontsize=10)
            axes[i].axis("off")
        plt.tight_layout()
        plt.show()

### 📝 **期末報告可解釋性論述建議 (Findings & Interpretation Guide)**
於報告 Slides 撰寫誤差分析（Error Analysis）時，可依據上述診斷結果結構化歸納：
1. **高召回優勢類別 (High Recall Classes)**：如 `traffic_light` 與 `crosswalk`，因特徵輪廓固定、色彩對比強烈，通常具備最高召回率。
2. **主要漏檢原因 (False Negatives Breakdown)**：
   * **尺度極小 (Scale variation)**：遠距離之小型標誌 (`Sign_limit`, `Sign_follow`) 像素過小，骨幹下採樣後特徵丟失。
   * **遮擋與複雜背景 (Occlusion & Clutter)**：受行道樹、逆光或車體遮擋時較易發生漏抓。
3. **虛警與類別混淆 (False Positives & Mismatch)**：
   * 特徵相似標誌之間的混淆（如藍底箭頭與圓形標誌）。
   * 背景招牌文字或建築輪廓被誤判為特定標誌。


# **Step 9: 儲存權重與期末成果打包 (Save to Google Drive)**
自動將各組權重、指標圖表、測試評估報告與推論成果打包壓縮至 Google Drive 備份，供期末報告投影片使用。


In [ ]:
import glob, os
%cd /content/workspace/yolov7
!mkdir -p /content/drive/MyDrive/NCHU_YOLO_Project_Traffic_Facility_6_classes/results

exp4_weights = sorted(glob.glob("runs/train/exp4_yolov7_tiny_tuned*/weights/best.pt"), key=os.path.getmtime)
if exp4_weights:
    !cp {exp4_weights[-1]} /content/drive/MyDrive/NCHU_YOLO_Project_Traffic_Facility_6_classes/results/best_model_exp4.pt

# 打包所有實驗圖表、測試集評估成果與推論文字標籤（先移除舊檔避免殘留）
!rm -f /content/drive/MyDrive/NCHU_YOLO_Project_Traffic_Facility_6_classes/results_summary.zip
!zip -r /content/drive/MyDrive/NCHU_YOLO_Project_Traffic_Facility_6_classes/results_summary.zip \
  runs/train/exp*/results.png \
  runs/train/exp*/confusion_matrix.png \
  runs/train/exp*/weights/best.pt \
  runs/test/exp*/confusion_matrix.png \
  runs/test/exp*/test_batch0_*.jpg \
  runs/test/exp*/*.png \
  runs/detect/

print("🎉 實驗指標、可解釋性圖表與預測文字檔已成功備份至 Google Drive (NCHU_YOLO_Project_Traffic_Facility_6_classes/)！")